In [24]:
!pip install scikit-posthocs


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [26]:
import pandas as pd
from scipy.stats import friedmanchisquare
from scipy.stats import wilcoxon
from itertools import combinations
import scikit_posthocs as sp

# WhatsApp

## Friendman

In [3]:
from scipy.stats import friedmanchisquare
import pandas as pd

# =====================================================
# CARREGAR RESULTADOS
# =====================================================

bow_wpp = pd.read_csv("./resultados/ml-bow-uni_gram_wpp_fold5_resultados.csv")

tfidf_wpp = pd.read_csv("./resultados/ml-tfidf-bi_gram_wpp_fold5_resultados.csv")

bow_dic_9_wpp = pd.read_csv(
    "./resultados/dic_9_vetorez_ml-bow-tri_gram_wpp_fold5_resultados.csv"
)

bow_dic_842_wpp = pd.read_csv(
    "./resultados/dic_842_vetorez_ml-bow-bi_gram_wpp_fold5_resultados.csv"
)

tfidf_dic_9_wpp = pd.read_csv(
    "./resultados/dic_9_vetorez_ml-tfidf-tri_gram_wpp_fold5_resultados.csv"
)

tfidf_dic_842_wpp = pd.read_csv(
    "./resultados/dic_842_vetorez_ml-tfidf-uni_gram_wpp_fold5_resultados.csv"
)

dic_9_wpp = pd.read_csv(
    "./resultados/dic_9_ml-dic_wpp_fold5_resultados.csv"
)

dic_842_wpp = pd.read_csv(
    "./resultados/dic_842_ml-dic_wpp_fold5_resultados.csv"
)

bert_wpp = pd.read_csv(
    "./resultados/bert_embedding_fold5_wpp.csv"
)

llm_wpp = pd.read_csv(
    "./resultados/llm_role_prompting_runs_wpp.csv"
)

# =====================================================
# PEGAR F1 DOS FOLDS
# =====================================================

bow_wpp_f1 = bow_wpp["f1"]

tfidf_wpp_f1 = tfidf_wpp["f1"]

bow_dic_9_wpp_f1 = bow_dic_9_wpp["f1"]

bow_dic_842_wpp_f1 = bow_dic_842_wpp["f1"]

tfidf_dic_9_wpp_f1 = tfidf_dic_9_wpp["f1"]

tfidf_dic_842_wpp_f1 = tfidf_dic_842_wpp["f1"]

dic_9_wpp_f1 = dic_9_wpp["f1"]

dic_842_wpp_f1 = dic_842_wpp["f1"]

bert_wpp_f1 = bert_wpp["f1"]

llm_wpp_f1 = llm_wpp["f1"]

# =====================================================
# TESTE DE FRIEDMAN
# =====================================================

stat, p = friedmanchisquare(
    bow_wpp_f1,
    tfidf_wpp_f1,
    bow_dic_9_wpp_f1,
    bow_dic_842_wpp_f1,
    tfidf_dic_9_wpp_f1,
    tfidf_dic_842_wpp_f1,
    dic_9_wpp_f1,
    dic_842_wpp_f1,
    bert_wpp_f1,
    llm_wpp_f1
)

print("\n========== TESTE DE FRIEDMAN ==========")

print(f"Statistic: {stat:.4f}")

print(f"P-value : {p:.6f}")

if p < 0.05:
    print("\nDiferença significativa entre modelos")

else:
    print("\nSem diferença significativa")


========== TESTE DE FRIEDMAN ==========
Statistic: 36.0545
P-value : 0.000039

Diferença significativa entre modelos


## wilcoxon

In [14]:
# =====================================================
# ORGANIZAR MODELOS
# =====================================================

models = {

    "BoW": bow_wpp["f1"],

    "TF-IDF": tfidf_wpp["f1"],

    "BoW + Dic9": bow_dic_9_wpp["f1"],

    "BoW + Dic842": bow_dic_842_wpp["f1"],

    "TF-IDF + Dic9": tfidf_dic_9_wpp["f1"],

    "TF-IDF + Dic842": tfidf_dic_842_wpp["f1"],

    "Dic9": dic_9_wpp["f1"],

    "Dic842": dic_842_wpp["f1"],

    "BERT": bert_wpp["f1"],

    "LLM": llm_wpp["f1"]
}

# =====================================================
# TODAS AS COMBINAÇÕES
# =====================================================

pairs = list(combinations(models.keys(), 2))

# =====================================================
# CORREÇÃO BONFERRONI
# =====================================================

n_comparisons = len(pairs)

alpha_original = 0.05

alpha_bonferroni = alpha_original / n_comparisons

print("\n====================================")
print("CORREÇÃO BONFERRONI")
print("====================================")

print(f"Número de comparações: {n_comparisons}")

print(f"Alpha original: {alpha_original}")

print(f"Alpha corrigido: {alpha_bonferroni:.6f}")

# =====================================================
# WILCOXON
# =====================================================

results = []

print("\n====================================")
print("TESTES DE WILCOXON")
print("====================================")

for model1, model2 in pairs:

    scores1 = models[model1]

    scores2 = models[model2]

    stat, p = wilcoxon(scores1, scores2)

    significant = p < alpha_bonferroni

    results.append({
        "model_1": model1,
        "model_2": model2,
        "statistic": stat,
        "p_value": p,
        "significant": significant
    })

    print(f"\n{model1} vs {model2}")

    print(f"Statistic: {stat:.4f}")

    print(f"P-value : {p:.6f}")

    if significant:
        print("Diferença significativa")

    else:
        print("Sem diferença significativa")

# =====================================================
# DATAFRAME FINAL
# =====================================================

results_df = pd.DataFrame(results)

# =====================================================
# ORDENAR PELO P-VALUE
# =====================================================

results_df = results_df.sort_values(
    by="p_value"
)

print("\n====================================")
print("RESULTADOS ORDENADOS")
print("====================================")

print(results_df)

# =====================================================
# SALVAR CSV
# =====================================================

results_df.to_csv(
    "wilcoxon_all_pairs_wpp.csv",
    index=False
)

print("\nCSV salvo com sucesso!")


CORREÇÃO BONFERRONI
Número de comparações: 45
Alpha original: 0.05
Alpha corrigido: 0.001111

TESTES DE WILCOXON

BoW vs TF-IDF
Statistic: 5.0000
P-value : 0.625000
Sem diferença significativa

BoW vs BoW + Dic9
Statistic: 4.0000
P-value : 0.437500
Sem diferença significativa

BoW vs BoW + Dic842
Statistic: 0.0000
P-value : 0.062500
Sem diferença significativa

BoW vs TF-IDF + Dic9
Statistic: 7.0000
P-value : 1.000000
Sem diferença significativa

BoW vs TF-IDF + Dic842
Statistic: 4.0000
P-value : 0.437500
Sem diferença significativa

BoW vs Dic9
Statistic: 0.0000
P-value : 0.062500
Sem diferença significativa

BoW vs Dic842
Statistic: 0.0000
P-value : 0.062500
Sem diferença significativa

BoW vs BERT
Statistic: 0.0000
P-value : 0.062500
Sem diferença significativa

BoW vs LLM
Statistic: 1.0000
P-value : 0.125000
Sem diferença significativa

TF-IDF vs BoW + Dic9
Statistic: 7.0000
P-value : 1.000000
Sem diferença significativa

TF-IDF vs BoW + Dic842
Statistic: 1.0000
P-value : 0.125000

## Nemenyi

In [31]:


# =====================================================
# DATAFRAME DOS MODELOS
# =====================================================

df_nemenyi = pd.DataFrame({

    "BoW": bow_wpp["f1"],

    "TF-IDF": tfidf_wpp["f1"],

    "BoW + Dic9": bow_dic_9_wpp["f1"],

    "BoW + Dic842": bow_dic_842_wpp["f1"],

    "TF-IDF + Dic9": tfidf_dic_9_wpp["f1"],

    "TF-IDF + Dic842": tfidf_dic_842_wpp["f1"],

    "Dic9": dic_9_wpp["f1"],

    "Dic842": dic_842_wpp["f1"],

    "BERT": bert_wpp["f1"],

    "LLM": llm_wpp["f1"]
})

# =====================================================
# TESTE DE NEMENYI
# =====================================================

nemenyi = sp.posthoc_nemenyi_friedman(
    df_nemenyi
)

print("\n====================================")
print("NEMENYI")
print("====================================")

print(nemenyi)

# =====================================================
# SALVAR CSV
# =====================================================

nemenyi.to_csv(
    "nemenyi_wpp.csv"
)

print("\nCSV salvo com sucesso!")


NEMENYI
                      BoW    TF-IDF  BoW + Dic9  BoW + Dic842  TF-IDF + Dic9  \
BoW              1.000000  1.000000    1.000000      0.963510       0.999994   
TF-IDF           1.000000  1.000000    1.000000      0.906921       0.999807   
BoW + Dic9       1.000000  1.000000    1.000000      0.864285       0.999319   
BoW + Dic842     0.963510  0.906921    0.864285      1.000000       0.998033   
TF-IDF + Dic9    0.999994  0.999807    0.999319      0.998033       1.000000   
TF-IDF + Dic842  0.999807  0.998033    0.995153      0.999807       1.000000   
Dic9             0.074109  0.129563    0.167184      0.000775       0.020181   
Dic842           0.390244  0.535342    0.610062      0.014062       0.167184   
BERT             0.461270  0.610062    0.682781      0.020181       0.212132   
LLM              0.989497  0.998033    0.999319      0.390244       0.906921   

                 TF-IDF + Dic842      Dic9    Dic842      BERT       LLM  
BoW                     0.999807  0

In [32]:


alpha = 0.05

print("\n====================================")
print("COMPARAÇÕES SIGNIFICATIVAS")
print("====================================")

for m1, m2 in combinations(
    nemenyi.columns,
    2
):

    p = nemenyi.loc[m1, m2]

    if p < alpha:

        print(f"{m1} vs {m2}")

        print(f"p-value = {p:.6f}\n")


COMPARAÇÕES SIGNIFICATIVAS
BoW + Dic842 vs Dic9
p-value = 0.000775

BoW + Dic842 vs Dic842
p-value = 0.014062

BoW + Dic842 vs BERT
p-value = 0.020181

TF-IDF + Dic9 vs Dic9
p-value = 0.020181

TF-IDF + Dic842 vs Dic9
p-value = 0.009665



# Telegram

## Friendman

In [17]:

# =====================================================
# CARREGAR RESULTADOS
# =====================================================

bow_telegram = pd.read_csv("./resultados/ml-bow-uni_gram_telegram_fold5_resultados.csv")

tfidf_telegram = pd.read_csv("./resultados/ml-tfidf-bi_gram_telegram_fold5_resultados.csv")

bow_dic_9_telegram = pd.read_csv(
    "./resultados/dic_9_vetorez_ml-bow-tri_gram_telegram_fold5_resultados.csv"
)

bow_dic_842_telegram = pd.read_csv(
    "./resultados/dic_842_vetorez_ml-bow-tri_gram_telegram_fold5_resultados.csv"
)

tfidf_dic_9_telegram = pd.read_csv(
    "./resultados/dic_9_vetorez_ml-tfidf-tri_gram_telegram_fold5_resultados.csv"
)

tfidf_dic_842_telegram = pd.read_csv(
    "./resultados/dic_842_vetorez_ml-tfidf-tri_gram_telegram_fold5_resultados.csv"
)

dic_9_telegram = pd.read_csv(
    "./resultados/dic_9_ml-dic_telegram_fold5_resultados.csv"
)

dic_842_telegram = pd.read_csv(
    "./resultados/dic_842_ml-dic_telegram_fold5_resultados.csv"
)

bert_telegram = pd.read_csv(
    "./resultados/bert_embedding_fold5_telegram.csv"
)

llm_telegram = pd.read_csv(
    "./resultados/llm_role_prompting_runs_telegram.csv"
)

# =====================================================
# PEGAR F1 DOS FOLDS
# =====================================================

bow_telegram_f1 = bow_telegram["f1"]

tfidf_telegram_f1 = tfidf_telegram["f1"]

bow_dic_9_telegram_f1 = bow_dic_9_telegram["f1"]

bow_dic_842_telegram_f1 = bow_dic_842_telegram["f1"]

tfidf_dic_9_telegram_f1 = tfidf_dic_9_telegram["f1"]

tfidf_dic_842_telegram_f1 = tfidf_dic_842_telegram["f1"]

dic_9_telegram_f1 = dic_9_telegram["f1"]

dic_842_telegram_f1 = dic_842_telegram["f1"]

bert_telegram_f1 = bert_telegram["f1"]

llm_telegram_f1 = llm_telegram["f1"]

# =====================================================
# TESTE DE FRIEDMAN
# =====================================================

stat, p = friedmanchisquare(
    bow_telegram_f1,
    tfidf_telegram_f1,
    bow_dic_9_telegram_f1,
    bow_dic_842_telegram_f1,
    tfidf_dic_9_telegram_f1,
    tfidf_dic_842_telegram_f1,
    dic_9_telegram_f1,
    dic_842_telegram_f1,
    bert_telegram_f1,
    llm_telegram_f1
)

print("\n========== TESTE DE FRIEDMAN ==========")

print(f"Statistic: {stat:.4f}")

print(f"P-value : {p:.6f}")

if p < 0.05:
    print("\nDiferença significativa entre modelos")

else:
    print("\nSem diferença significativa")


========== TESTE DE FRIEDMAN ==========
Statistic: 37.5947
P-value : 0.000021

Diferença significativa entre modelos


## wilcoxon

In [ ]:
# =====================================================
# ORGANIZAR MODELOS
# =====================================================

models = {

    "BoW": bow_telegram["f1"],

    "TF-IDF": tfidf_telegram["f1"],

    "BoW + Dic9": bow_dic_9_telegram["f1"],

    "BoW + Dic842": bow_dic_842_telegram["f1"],

    "TF-IDF + Dic9": tfidf_dic_9_telegram["f1"],

    "TF-IDF + Dic842": tfidf_dic_842_telegram["f1"],

    "Dic9": dic_9_telegram["f1"],

    "Dic842": dic_842_telegram["f1"],

    "BERT": bert_telegram["f1"],

    "LLM": llm_telegram["f1"]
}

# =====================================================
# TODAS AS COMBINAÇÕES
# =====================================================

pairs = list(combinations(models.keys(), 2))

# =====================================================
# CORREÇÃO BONFERRONI
# =====================================================

n_comparisons = len(pairs)

alpha_original = 0.05

alpha_bonferroni = alpha_original / n_comparisons

print("\n====================================")
print("CORREÇÃO BONFERRONI")
print("====================================")

print(f"Número de comparações: {n_comparisons}")

print(f"Alpha original: {alpha_original}")

print(f"Alpha corrigido: {alpha_bonferroni:.6f}")

# =====================================================
# WILCOXON
# =====================================================

results = []

print("\n====================================")
print("TESTES DE WILCOXON")
print("====================================")

for model1, model2 in pairs:

    scores1 = models[model1]

    scores2 = models[model2]

    stat, p = wilcoxon(scores1, scores2)

    significant = p < alpha_bonferroni

    results.append({
        "model_1": model1,
        "model_2": model2,
        "statistic": stat,
        "p_value": p,
        "significant": significant
    })

    print(f"\n{model1} vs {model2}")

    print(f"Statistic: {stat:.4f}")

    print(f"P-value : {p:.6f}")

    if significant:
        print("Diferença significativa")

    else:
        print("Sem diferença significativa")

# =====================================================
# DATAFRAME FINAL
# =====================================================

results_df = pd.DataFrame(results)

# =====================================================
# ORDENAR PELO P-VALUE
# =====================================================

results_df = results_df.sort_values(
    by="p_value"
)

print("\n====================================")
print("RESULTADOS ORDENADOS")
print("====================================")

print(results_df)

# =====================================================
# SALVAR CSV
# =====================================================

results_df.to_csv(
    "wilcoxon_all_pairs_telegram.csv",
    index=False
)

print("\nCSV salvo com sucesso!")


CORREÇÃO BONFERRONI
Número de comparações: 45
Alpha original: 0.05
Alpha corrigido: 0.001111

TESTES DE WILCOXON

BoW vs TF-IDF
Statistic: 3.0000
P-value : 0.312500
Sem diferença significativa

BoW vs BoW + Dic9
Statistic: 7.0000
P-value : 1.000000
Sem diferença significativa

BoW vs BoW + Dic842
Statistic: 5.0000
P-value : 0.625000
Sem diferença significativa

BoW vs TF-IDF + Dic9
Statistic: 1.0000
P-value : 0.125000
Sem diferença significativa

BoW vs TF-IDF + Dic842
Statistic: 1.0000
P-value : 0.125000
Sem diferença significativa

BoW vs Dic9
Statistic: 0.0000
P-value : 0.062500
Sem diferença significativa

BoW vs Dic842
Statistic: 0.0000
P-value : 0.062500
Sem diferença significativa

BoW vs BERT
Statistic: 0.0000
P-value : 0.062500
Sem diferença significativa

BoW vs LLM
Statistic: 0.0000
P-value : 0.062500
Sem diferença significativa

TF-IDF vs BoW + Dic9
Statistic: 5.0000
P-value : 0.625000
Sem diferença significativa

TF-IDF vs BoW + Dic842
Statistic: 4.0000
P-value : 0.437500

c:\Users\Melissa Felipe\.conda\envs\gpu_env2\lib\site-packages\scipy\stats\_wilcoxon.py:199: UserWarning: Sample size too small for normal approximation.
  temp = _wilcoxon_iv(x, y, zero_method, correction, alternative, method, axis)


## Nemenyi

In [28]:


# =====================================================
# DATAFRAME DOS MODELOS
# =====================================================

df_nemenyi = pd.DataFrame({

    "BoW": bow_telegram["f1"],

    "TF-IDF": tfidf_telegram["f1"],

    "BoW + Dic9": bow_dic_9_telegram["f1"],

    "BoW + Dic842": bow_dic_842_telegram["f1"],

    "TF-IDF + Dic9": tfidf_dic_9_telegram["f1"],

    "TF-IDF + Dic842": tfidf_dic_842_telegram["f1"],

    "Dic9": dic_9_telegram["f1"],

    "Dic842": dic_842_telegram["f1"],

    "BERT": bert_telegram["f1"],

    "LLM": llm_telegram["f1"]
})

# =====================================================
# TESTE DE NEMENYI
# =====================================================

nemenyi = sp.posthoc_nemenyi_friedman(
    df_nemenyi
)

print("\n====================================")
print("NEMENYI")
print("====================================")

print(nemenyi)

# =====================================================
# SALVAR CSV
# =====================================================

nemenyi.to_csv(
    "nemenyi_telegram.csv"
)

print("\nCSV salvo com sucesso!")


NEMENYI
                      BoW    TF-IDF  BoW + Dic9  BoW + Dic842  TF-IDF + Dic9  \
BoW              1.000000  0.989497    1.000000      0.999907       0.989497   
TF-IDF           0.989497  1.000000    0.998816      0.999983       1.000000   
BoW + Dic9       1.000000  0.998816    1.000000      1.000000       0.998816   
BoW + Dic842     0.999907  0.999983    1.000000      1.000000       0.999983   
TF-IDF + Dic9    0.989497  1.000000    0.998816      0.999983       1.000000   
TF-IDF + Dic842  0.979527  1.000000    0.996857      0.999907       1.000000   
Dic9             0.167184  0.006554    0.085726      0.033789       0.006554   
Dic842           0.461270  0.039833    0.293527      0.147482       0.039833   
BERT             0.864285  0.212132    0.717554      0.498067       0.212132   
LLM              0.963510  0.390244    0.886843      0.717554       0.390244   

                 TF-IDF + Dic842      Dic9    Dic842      BERT       LLM  
BoW                     0.979527  0

In [29]:


alpha = 0.05

print("\n====================================")
print("COMPARAÇÕES SIGNIFICATIVAS")
print("====================================")

for m1, m2 in combinations(
    nemenyi.columns,
    2
):

    p = nemenyi.loc[m1, m2]

    if p < alpha:

        print(f"{m1} vs {m2}")

        print(f"p-value = {p:.6f}\n")


COMPARAÇÕES SIGNIFICATIVAS
TF-IDF vs Dic9
p-value = 0.006554

TF-IDF vs Dic842
p-value = 0.039833

BoW + Dic842 vs Dic9
p-value = 0.033789

TF-IDF + Dic9 vs Dic9
p-value = 0.006554

TF-IDF + Dic9 vs Dic842
p-value = 0.039833

TF-IDF + Dic842 vs Dic9
p-value = 0.004386

TF-IDF + Dic842 vs Dic842
p-value = 0.028557



# WhatsApp vs Telegram

In [23]:
import pandas as pd

from itertools import combinations

from scipy.stats import wilcoxon

# =====================================================
# CARREGAR RESULTADOS WHATSAPP
# =====================================================

bow_wpp = pd.read_csv(
    "./resultados/ml-bow-uni_gram_wpp_fold5_resultados.csv"
)

tfidf_wpp = pd.read_csv(
    "./resultados/ml-tfidf-bi_gram_wpp_fold5_resultados.csv"
)

bow_dic_9_wpp = pd.read_csv(
    "./resultados/dic_9_vetorez_ml-bow-tri_gram_wpp_fold5_resultados.csv"
)

bow_dic_842_wpp = pd.read_csv(
    "./resultados/dic_842_vetorez_ml-bow-bi_gram_wpp_fold5_resultados.csv"
)

tfidf_dic_9_wpp = pd.read_csv(
    "./resultados/dic_9_vetorez_ml-tfidf-tri_gram_wpp_fold5_resultados.csv"
)

tfidf_dic_842_wpp = pd.read_csv(
    "./resultados/dic_842_vetorez_ml-tfidf-uni_gram_wpp_fold5_resultados.csv"
)

dic_9_wpp = pd.read_csv(
    "./resultados/dic_9_ml-dic_wpp_fold5_resultados.csv"
)

dic_842_wpp = pd.read_csv(
    "./resultados/dic_842_ml-dic_wpp_fold5_resultados.csv"
)

bert_wpp = pd.read_csv(
    "./resultados/bert_embedding_fold5_wpp.csv"
)

llm_wpp = pd.read_csv(
    "./resultados/llm_role_prompting_runs_wpp.csv"
)

# =====================================================
# CARREGAR RESULTADOS TELEGRAM
# =====================================================

bow_telegram = pd.read_csv(
    "./resultados/ml-bow-uni_gram_telegram_fold5_resultados.csv"
)

tfidf_telegram = pd.read_csv(
    "./resultados/ml-tfidf-bi_gram_telegram_fold5_resultados.csv"
)

bow_dic_9_telegram = pd.read_csv(
    "./resultados/dic_9_vetorez_ml-bow-tri_gram_telegram_fold5_resultados.csv"
)

bow_dic_842_telegram = pd.read_csv(
    "./resultados/dic_842_vetorez_ml-bow-tri_gram_telegram_fold5_resultados.csv"
)

tfidf_dic_9_telegram = pd.read_csv(
    "./resultados/dic_9_vetorez_ml-tfidf-tri_gram_telegram_fold5_resultados.csv"
)

tfidf_dic_842_telegram = pd.read_csv(
    "./resultados/dic_842_vetorez_ml-tfidf-tri_gram_telegram_fold5_resultados.csv"
)

dic_9_telegram = pd.read_csv(
    "./resultados/dic_9_ml-dic_telegram_fold5_resultados.csv"
)

dic_842_telegram = pd.read_csv(
    "./resultados/dic_842_ml-dic_telegram_fold5_resultados.csv"
)

bert_telegram = pd.read_csv(
    "./resultados/bert_embedding_fold5_telegram.csv"
)

llm_telegram = pd.read_csv(
    "./resultados/llm_role_prompting_runs_telegram.csv"
)

# =====================================================
# ORGANIZAR MODELOS
# =====================================================

models = {

    "BoW": (
        bow_wpp["f1"],
        bow_telegram["f1"]
    ),

    "TF-IDF": (
        tfidf_wpp["f1"],
        tfidf_telegram["f1"]
    ),

    "BoW + Dic9": (
        bow_dic_9_wpp["f1"],
        bow_dic_9_telegram["f1"]
    ),

    "BoW + Dic842": (
        bow_dic_842_wpp["f1"],
        bow_dic_842_telegram["f1"]
    ),

    "TF-IDF + Dic9": (
        tfidf_dic_9_wpp["f1"],
        tfidf_dic_9_telegram["f1"]
    ),

    "TF-IDF + Dic842": (
        tfidf_dic_842_wpp["f1"],
        tfidf_dic_842_telegram["f1"]
    ),

    "Dic9": (
        dic_9_wpp["f1"],
        dic_9_telegram["f1"]
    ),

    "Dic842": (
        dic_842_wpp["f1"],
        dic_842_telegram["f1"]
    ),

    "BERT": (
        bert_wpp["f1"],
        bert_telegram["f1"]
    ),

    "LLM": (
        llm_wpp["f1"],
        llm_telegram["f1"]
    )
}

# =====================================================
# WILCOXON ENTRE DATASETS
# =====================================================

results = []

print("\n====================================")
print("WHATSAPP vs TELEGRAM")
print("====================================")

for model_name, (scores_wpp, scores_telegram) in models.items():

    stat, p = wilcoxon(
        scores_wpp,
        scores_telegram
    )

    significant = p < 0.05

    mean_wpp = scores_wpp.mean()

    std_wpp = scores_wpp.std()

    mean_telegram = scores_telegram.mean()

    std_telegram = scores_telegram.std()

    results.append({

        "model": model_name,

        "wpp_mean_f1": mean_wpp,

        "wpp_std_f1": std_wpp,

        "telegram_mean_f1": mean_telegram,

        "telegram_std_f1": std_telegram,

        "statistic": stat,

        "p_value": p,

        "significant": significant
    })

    print(f"\n{model_name}")

    print(
        f"WPP      : {mean_wpp:.4f} ± {std_wpp:.4f}"
    )

    print(
        f"Telegram : {mean_telegram:.4f} ± {std_telegram:.4f}"
    )

    print(f"Statistic: {stat:.4f}")

    print(f"P-value : {p:.6f}")

    if significant:
        print("Diferença significativa")

    else:
        print("Sem diferença significativa")

# =====================================================
# DATAFRAME FINAL
# =====================================================

results_df = pd.DataFrame(results)

# =====================================================
# ORDENAR PELO P-VALUE
# =====================================================

results_df = results_df.sort_values(
    by="p_value"
)

print("\n====================================")
print("RESULTADOS ORDENADOS")
print("====================================")

print(results_df)

# =====================================================
# SALVAR CSV
# =====================================================

results_df.to_csv(
    "wilcoxon_wpp_vs_telegram.csv",
    index=False
)

print("\nCSV salvo com sucesso!")


WHATSAPP vs TELEGRAM

BoW
WPP      : 0.8690 ± 0.0138
Telegram : 0.8527 ± 0.0105
Statistic: 2.0000
P-value : 0.187500
Sem diferença significativa

TF-IDF
WPP      : 0.8651 ± 0.0127
Telegram : 0.8571 ± 0.0081
Statistic: 2.0000
P-value : 0.187500
Sem diferença significativa

BoW + Dic9
WPP      : 0.8650 ± 0.0163
Telegram : 0.8541 ± 0.0141
Statistic: 4.0000
P-value : 0.437500
Sem diferença significativa

BoW + Dic842
WPP      : 0.8778 ± 0.0131
Telegram : 0.8540 ± 0.0113
Statistic: 1.0000
P-value : 0.125000
Sem diferença significativa

TF-IDF + Dic9
WPP      : 0.8691 ± 0.0146
Telegram : 0.8577 ± 0.0106
Statistic: 0.0000
P-value : 0.062500
Sem diferença significativa

TF-IDF + Dic842
WPP      : 0.8781 ± 0.0110
Telegram : 0.8585 ± 0.0123
Statistic: 1.0000
P-value : 0.125000
Sem diferença significativa

Dic9
WPP      : 0.7264 ± 0.0150
Telegram : 0.6333 ± 0.0296
Statistic: 0.0000
P-value : 0.062500
Sem diferença significativa

Dic842
WPP      : 0.8133 ± 0.0181
Telegram : 0.7064 ± 0.0151
Statis